# Task 3 - SARIMA (Kaggle/Colab)
Run SARIMA for the three required areas and save predictions + metrics.

In [ ]:
# Setup and data load
from pathlib import Path
import time
import numpy as np
import pandas as pd
from statsmodels.tsa.statespace.sarimax import SARIMAX

# Set this path to your parquet in Kaggle/Colab
DATA_PATH = Path("/kaggle/input/telecom-traffic/city_traffic.parquet")
if not DATA_PATH.exists():
    raise FileNotFoundError(f"Parquet not found at {DATA_PATH}")

OUTPUT_DIR = Path("outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

df = pd.read_parquet(DATA_PATH)
df = df.sort_values(["square_id", "time_interval"])
df["time_interval"] = pd.to_datetime(df["time_interval"], utc=True)

def get_area_series(data: pd.DataFrame, square_id: int) -> pd.Series:
    s = data[data["square_id"] == square_id].sort_values("time_interval")
    return s.set_index("time_interval")["internet_traffic"]

def split_train_test(series: pd.Series):
    train_end = pd.Timestamp("2013-12-15 23:50:00", tz="UTC")
    test_start = pd.Timestamp("2013-12-16 00:00:00", tz="UTC")
    test_end = pd.Timestamp("2013-12-22 23:50:00", tz="UTC")
    train = series.loc[:train_end]
    test = series.loc[test_start:test_end]
    return train, test

target_squares = [df.groupby("square_id")["internet_traffic"].sum().idxmax(), 4159, 4556]

In [ ]:
# SARIMA training + evaluation
def evaluate_metrics(y_true: np.ndarray, y_pred: np.ndarray) -> dict:
    mae = float(np.mean(np.abs(y_true - y_pred)))
    mape = float(np.mean(np.abs((y_true - y_pred) / np.maximum(np.abs(y_true), 1e-8))) * 100.0)
    rmse = float(np.sqrt(np.mean((y_true - y_pred) ** 2)))
    return {"MAE": mae, "MAPE": mape, "RMSE": rmse}

sarima_results = []
for square_id in target_squares:
    series = get_area_series(df, int(square_id))
    train, test = split_train_test(series)

    start = time.perf_counter()
    model = SARIMAX(
        train,
        order=(1, 1, 1),
        seasonal_order=(1, 1, 1, 144),
        enforce_stationarity=False,
        enforce_invertibility=False,
    )
    fit = model.fit(disp=False)
    train_time = time.perf_counter() - start

    start = time.perf_counter()
    forecast = fit.forecast(steps=len(test))
    infer_time = time.perf_counter() - start

    y_true = test.values
    y_pred = forecast.values
    metrics = evaluate_metrics(y_true, y_pred)
    metrics.update({"square_id": int(square_id), "train_seconds": train_time, "infer_seconds": infer_time})
    sarima_results.append(metrics)

    out = pd.DataFrame({
        "time_interval": test.index,
        "y_true": y_true,
        "y_pred": y_pred,
        "model": "SARIMA",
        "square_id": int(square_id),
    })
    out.to_csv(OUTPUT_DIR / f"sarima_{int(square_id)}.csv", index=False)

pd.DataFrame(sarima_results).to_csv(OUTPUT_DIR / "metrics_sarima.csv", index=False)
pd.DataFrame(sarima_results)